# Workshop: Fusion Strategies on the PARLO Corpus
**Duration:** about 90 minutes (plus feature extraction, see Part 3).

This workshop is the kick-off for the PStA.
You will load the PARLO Dementia Corpus with `parlo-loader`, set up the participant-safe evaluation protocol, and compare the **fusion strategies** from the lecture.

The focus is on **understanding**, not on coding: all code cells run as they are.
Read the code, run it, and spend most of the time on the **discussion questions** (in your team).

To keep things fast and easy to interpret we only use the **audio modality** and two PARLO tasks:
- `pictdes`: describe a mountain scene (spontaneous, cognitively demanding speech, about one to two minutes)
- `pataka`: repeat "pataka" for ten seconds (diadochokinesis, a motor speech task with almost no linguistic content)

From every recording we extract two **neural representations**:
- **WavLM** (self-supervised speech model, Lab 02_03): *how* something is said, including phonetic and prosodic information
- **ECAPA-TDNN** (speaker embedding, Lab 02_03): *who* is speaking (voice timbre, age, gender, channel)

This gives a 2 × 2 grid of **sources** (task × representation).
In the PStA you will fuse audio and text, but the strategies are exactly the same.

**Target:** three-class classification HC / MCI / AD (the PStA challenge).
**Metric:** unweighted average recall (UAR), with five participant-level folds.

> **Legal reminder:** the PARLO data must not leave your machine.
> Do not upload audio, transcripts, features or notebook outputs (they can contain audio players!) to GitHub, cloud drives or online services.

## Fusion strategies at a glance
Fusion combines information from several modalities, feature sets or models.
The strategies differ in **where** the combination happens.

**Early fusion (feature level)**
- concatenate the feature vectors, e.g. $[x_{pictdes}; x_{pataka}]$, and train **one** classifier on the joint vector
- can model interactions between the sources
- needs all sources for every sample
- watch the scale and dimensionality: 768 WavLM vs. 192 ECAPA dimensions

**Late fusion (decision level)**
- train **one classifier per source** and combine their outputs: $p(c \mid x) = \sum_m w_m \, p_m(c \mid x_m)$
- weights $w_m$ tuned on training data only, or simple averaging
- modular, and a missing source can simply be skipped
- cannot model interactions; probabilities should be (reasonably) calibrated

**Stacking (learned late fusion)**
- the unimodal models produce **out-of-fold** probabilities
- a small meta-classifier, e.g. logistic regression, learns how to combine them
- the meta-classifier must **never** see predictions on data the base models were trained on, otherwise the evaluation leaks

**Intermediate / hybrid fusion**
- encode each source separately, then merge the **learned representations** (concatenation, a gated sum, cross-attention)
- the whole model is trained end to end
- the most flexible option and the most prone to overfitting on small data

| | Early | Late | Stacking | Hybrid (end to end) |
|---|---|---|---|---|
| Combination at | features | decisions | decisions, learned | learned representations |
| Models interactions | yes | no | no | yes |
| Data needed | low to medium | low | medium | high |
| Missing source | hard | easy | easy | hard |
| Typical risk | dimension imbalance | one weak model dilutes the strong one | leakage | overfitting |

**Pitfalls**
- always compare the fusion model with the **best single source**, fusion does not automatically win
- a much weaker model can hurt the fusion result
- fit scalers, PCA and fusion weights on the training folds only
- the participant-level split applies to every fusion variant
- start simple: concatenation plus logistic regression, then probability averaging, then anything fancier

# Part 0: Setup (5 min)
The corpus is **not** part of the course repository.
Set the environment variable `PARLO_ROOT` to the folder that contains `0metadata.csv`, `0metadata.cdc` and the folders `AD`, `HC` and `MCI`, for example:

```bash
export PARLO_ROOT=~/data/PARLO   # before starting Jupyter
```

Alternatively, edit the fallback path in the next cell.

In [ ]:
# run if needed (the course uv environment already contains everything):
#!pip install "parlo-loader[notebook,torch] @ git+https://github.com/patientzero/parlo-loader.git" torch transformers speechbrain scikit-learn pandas matplotlib

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import torch
from torch import nn
from tqdm.auto import tqdm
from IPython.display import display

from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay

from parlo import Corpus
from slp_lab_utils import SR, CACHE_DIR, cached, play

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', DEVICE)

PARLO_ROOT = Path(os.environ.get('PARLO_ROOT', '~/data/parlo_dataset/parlo_dataset')).expanduser()
if not (PARLO_ROOT / '0metadata.csv').exists():
    raise FileNotFoundError(f'No PARLO corpus at {PARLO_ROOT}: set PARLO_ROOT or edit the fallback path above')

TASKS = ['pictdes', 'pataka']

# Part 1: Loading PARLO with `parlo-loader` (15 min)
`Corpus(root)` only checks that the folder exists.
On the first table access the loader reads the metadata, the CHAT transcripts and the WAV **headers** (no audio samples) and stores everything as Parquet files in `<root>/.parlo-cache`.
Later accesses reuse this cache, so the first cell takes a few seconds and every later one is fast.

All tables are **lazy** [Polars](https://docs.pola.rs/) `LazyFrame`s: you describe a query and `.collect()` executes it.

In [ ]:
corpus = Corpus(PARLO_ROOT)
print(corpus.cache_status())

# the loader reports problems instead of silently dropping files
corpus.validation_report.group_by('severity', 'code').len().sort('len', descending=True).collect()

## Participants and recordings
There is one row per **participant** (with all metadata columns as strings) and one row per **recording** (participant × task).

In [ ]:
display(corpus.participants.filter(pl.col('has_recordings')).group_by('diagnosis').len().sort('diagnosis').collect())

corpus.recordings.group_by('task').agg(
    pl.len().alias('recordings'),
    (pl.col('duration_ms').median() / 1000).round(1).alias('median duration [s]'),
).sort('task').collect()

## A view on our two tasks
`corpus.filter(...)` returns a **view**.
All tables of a view (`participants`, `recordings`, `utterances`, `words`) are restricted consistently.

In [ ]:
view = corpus.filter(task=TASKS)

view.recordings.group_by('task', 'diagnosis').agg(
    pl.len().alias('n'),
    (pl.col('duration_ms').mean() / 1000).round(1).alias('mean duration [s]'),
).sort('task', 'diagnosis').collect()

Not every participant has every task.
`complete_cases(tasks)` keeps only participants for whom **all** listed tasks are available (as files; it does not check the quality of the recording).

In [ ]:
complete = view.complete_cases(TASKS)
n_any = view.participants.select(pl.len()).collect().item()
n_complete = complete.participants.select(pl.len()).collect().item()
print(f'participants with at least one of {TASKS}: {n_any}')
print(f'participants with both tasks:            {n_complete}')

## Listening to the data
`corpus.load_audio(row)` is the only place where audio is decoded.
By default you get the native sample rate and all channels; we ask for mono 16 kHz, as needed by the neural models.

In [ ]:
# one example per task (the same participant)
pid = complete.participants.select('participant_id').collect().item(0, 0)
for row in corpus.participant(pid).filter(task=TASKS).recordings.sort('task').collect().iter_rows(named=True):
    audio = corpus.load_audio(row, mono=True, sample_rate=SR)
    print(f"{row['task']:8s} {row['diagnosis']}  {audio.samples.shape[0] / SR:5.1f} s")
    display(play(audio.samples, audio.sample_rate))

**Questions**
1. Which abilities do `pictdes` and `pataka` test? Which kind of impairment (cognitive, linguistic, motor) would you expect to show up in which task?
2. Why does the loader distinguish between "available" and "complete" participants? What does it mean for early fusion that some participants miss a task?
3. Look at the class distribution. Why is accuracy a bad metric here, and what is the chance level of UAR for three classes?

# Part 2: The Evaluation Protocol (10 min)
## Targets
`corpus.targets()` turns the diagnosis into an integer target and documents the mapping.
For binary tasks you could use `targets(labels=['HC', 'AD'])` or `targets('hc_vs_impaired')`, and for regression `targets('mmse')`.

In [ ]:
targets = corpus.targets()                      # HC=0, MCI=1, AD=2
CLASSES = list(targets.mapping)
print(targets.mapping)

y = targets.participants.select('participant_id', 'target').collect().to_pandas().set_index('participant_id')['target']
y.value_counts().sort_index().rename(index=dict(enumerate(CLASSES)))

## Participant-level, stratified k-fold
`corpus.kfold()` assigns every **participant** to exactly one test fold (stratified by diagnosis) **before** the data is expanded to tasks or recordings.
That is the grouped k-fold you know from the lecture: no participant can appear in train and test, no matter how many recordings they have.

We create the folds once on the **whole corpus** and reuse them for every task, feature set and fusion model.
If we created separate folds per task, the systems would be tested on different participants, and fusing them would leak (a participant could be in the training fold of one model and the test fold of the other).

In [ ]:
FOLDS = corpus.kfold(folds=5, seed=42)

for i, split in enumerate(FOLDS):
    train_ids, test_ids = split.assignments['train'], split.assignments['test']
    assert not set(train_ids) & set(test_ids)           # participant-disjoint
    counts = y.reindex(test_ids).value_counts().sort_index().tolist()
    print(f'fold {i}: {len(train_ids):3d} train, {len(test_ids):3d} test participants, test per class {dict(zip(CLASSES, counts))}')

# the same manifest filters any participant-keyed table, here the test recordings of fold 0 for both tasks
FOLDS[0].apply(view.recordings, 'test').group_by('task').len().collect()

Manifests can be saved with `split.save('fold0.json')` and loaded again with `SplitManifest.load(...)`.
In your PStA, save the folds once and use them for **all** experiments, so every number in your paper is comparable.

## One evaluation function for every system
Every system in this notebook is a function `fit_predict(train_ids, test_ids)` that trains on the training participants and returns a table of class probabilities for (some of) the test participants.
`run_cv` calls it for all five folds and collects the **out-of-fold** probabilities: every participant gets exactly one prediction, from the model that did not see them during training.

To compare systems fairly, we compute the UAR on the **same participants** for every system: those with both tasks.
We report the mean and standard deviation over the five folds.

In [ ]:
results, oof = {}, {}
EVAL_IDS = pd.Index(complete.participants.select('participant_id').collect().to_series().to_list())


def uar(ids, proba):
    return balanced_accuracy_score(y[ids], proba.loc[ids].to_numpy().argmax(1))


def run_cv(name, fit_predict, verbose=True):
    folds = []
    for split in FOLDS:
        proba = fit_predict(pd.Index(split.assignments['train']), pd.Index(split.assignments['test']))
        folds.append(proba)
    proba = pd.concat(folds)
    fold_uars = [uar(p.index.intersection(EVAL_IDS), p) for p in folds]
    oof[name] = proba
    results[name] = {'UAR': np.mean(fold_uars), 'std': np.std(fold_uars), 'n predicted': len(proba)}
    if verbose:
        print(f'{name:45s} UAR = {np.mean(fold_uars):.3f} ± {np.std(fold_uars):.3f}   ({len(proba)} participants predicted)')
    return proba

**Questions**
1. Why must the split be on participant level, even though in this notebook every participant has only one recording per task?
2. Why do we create the folds on the whole corpus and not separately for the `pictdes` and `pataka` views?
3. We report UAR on the participants with both tasks only. What would be unfair if every system were evaluated on all participants it can predict?

# Part 3: Neural Features (10 min, extraction runs once)
We freeze two pre-trained networks and use them as feature extractors (as in Lab 02_03).

**WavLM base plus:** we keep the mean-pooled hidden states of **all** layers and use a middle layer by default.
`pictdes` recordings are long; a Transformer's memory grows quadratically with the length, so we cut the signal into 10 s chunks, pool every chunk and average the chunks (weighted by their length).

**ECAPA-TDNN:** a speaker verification model (192 dimensions) that already pools over time.
We use the same chunking for consistency.

The extraction takes a few minutes per task and model on a laptop (faster on a GPU or Apple MPS).
Results are cached in `~/.cache/slp_labs` (outside the repository; it is derived from the corpus, so keep it local as well).

In [ ]:
from transformers import AutoFeatureExtractor, AutoModel
from speechbrain.inference.speaker import EncoderClassifier

CHUNK = 10 * SR        # 10 s chunks
WAVLM = 'microsoft/wavlm-base-plus'


def chunks(samples):
    starts = range(0, max(len(samples) - SR, 1), CHUNK)        # drop a final chunk shorter than 1 s
    return [samples[s:s + CHUNK] for s in starts]


def recordings_of(task):
    return corpus.filter(task=task).recordings.sort('participant_id').collect()


@torch.inference_mode()
def wavlm_features(task):
    extractor = AutoFeatureExtractor.from_pretrained(WAVLM)
    model = AutoModel.from_pretrained(WAVLM).to(DEVICE).eval()
    rows = recordings_of(task)
    out = []
    for row in tqdm(rows.iter_rows(named=True), total=rows.height, desc=f'WavLM {task}'):
        samples = corpus.load_audio(row, mono=True, sample_rate=SR).samples
        pooled, weights = [], []
        for chunk in chunks(samples):
            inputs = extractor(chunk, sampling_rate=SR, return_tensors='pt').to(DEVICE)
            hidden = model(**inputs, output_hidden_states=True).hidden_states
            pooled.append(torch.stack([h[0].mean(dim=0) for h in hidden]).cpu().numpy())   # (13, 768)
            weights.append(len(chunk))
        out.append(np.average(pooled, axis=0, weights=weights))
    return rows['participant_id'].to_list(), np.stack(out)                                  # (n, 13, 768)


@torch.inference_mode()
def ecapa_features(task):
    ecapa = EncoderClassifier.from_hparams('speechbrain/spkrec-ecapa-voxceleb', savedir=str(CACHE_DIR / 'ecapa'),
                                           run_opts={'device': DEVICE})
    rows = recordings_of(task)
    out = []
    for row in tqdm(rows.iter_rows(named=True), total=rows.height, desc=f'ECAPA {task}'):
        samples = corpus.load_audio(row, mono=True, sample_rate=SR).samples
        emb = [ecapa.encode_batch(torch.tensor(c)[None]).squeeze().cpu().numpy() for c in chunks(samples)]
        out.append(np.average(emb, axis=0, weights=[len(c) for c in chunks(samples)]))
    return rows['participant_id'].to_list(), np.stack(out)                                  # (n, 192)


wavlm, ecapa = {}, {}
for task in TASKS:
    wavlm[task] = cached(f'parlo_wavlm_{task}', lambda: wavlm_features(task))
    ecapa[task] = cached(f'parlo_ecapa_{task}', lambda: ecapa_features(task))
    print(f'{task:8s} WavLM {wavlm[task][1].shape}   ECAPA {ecapa[task][1].shape}')

We collect the four sources as tables with one row per participant (index = participant ID).
The index makes the alignment explicit: fusion only ever combines rows of the **same** participant.

In [ ]:
WAVLM_LAYER = 6        # a middle layer; try others (see the questions)

sources = {}
for task in TASKS:
    ids, H = wavlm[task]
    sources[f'{task}·WavLM'] = pd.DataFrame(H[:, WAVLM_LAYER], index=ids)
    ids, E = ecapa[task]
    sources[f'{task}·ECAPA'] = pd.DataFrame(E, index=ids)

pd.DataFrame({name: {'participants': len(X), 'dimensions': X.shape[1]} for name, X in sources.items()}).T

**Questions**
1. Why do we pool over time at all? Which information relevant for dementia (or for `pataka`) gets lost by mean pooling?
2. ECAPA was trained to recognise *speakers*. Why can it still be useful for a diagnosis? Why is it also a **risk** in a multi-centre corpus such as PARLO?
3. Which WavLM layer would you choose, and how would you choose it without looking at the test folds?

# Part 4: Single-Source Baselines (10 min)
Every fusion model has to beat the **best single source**.
We use the same simple, well-regularised classifier everywhere: standardisation plus logistic regression with balanced class weights.
Logistic regression also gives reasonably calibrated probabilities, which we need for late fusion.

In [ ]:
def lr():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000, class_weight='balanced'))


def fit_lr(X, model=lr):
    # train on the training participants that have this source, predict the test participants that have it
    def fit_predict(train_ids, test_ids):
        train_ids, test_ids = X.index.intersection(train_ids), X.index.intersection(test_ids)
        clf = model().fit(X.loc[train_ids], y[train_ids])
        return pd.DataFrame(clf.predict_proba(X.loc[test_ids]), index=test_ids, columns=CLASSES)
    return fit_predict


for name, X in sources.items():
    run_cv(name, fit_lr(X))

## When can fusion help at all?
Fusion can only help if the sources make **different** errors.
We compare the out-of-fold predictions of the two WavLM models on the participants with both tasks.

In [ ]:
def error_overlap(a, b):
    correct = {n: oof[n].loc[EVAL_IDS].to_numpy().argmax(1) == y[EVAL_IDS].to_numpy() for n in (a, b)}
    return pd.crosstab(pd.Series(correct[a], name=f'{a} correct'), pd.Series(correct[b], name=f'{b} correct'), margins=True)

display(error_overlap('pictdes·WavLM', 'pataka·WavLM'))
display(error_overlap('pictdes·WavLM', 'pictdes·ECAPA'))

**Questions**
1. Which source is best? Is the ranking what you expected from Part 1, Question 1?
2. Look at the standard deviations. How large must a difference between two systems be before you believe it?
3. In the cross tables, which cells represent the potential of fusion? Which pair of sources looks more complementary: two tasks, or two representations of the same task?

# Part 5: Early Fusion (10 min)
Early fusion concatenates the feature vectors of the same participant and trains **one** classifier.
`join='inner'` keeps only participants who have all sources: early fusion cannot handle a missing source.

In [ ]:
def concat(*names):
    return pd.concat([sources[n].add_prefix(f'{n}:') for n in names], axis=1, join='inner')


def label(names):
    return 'all sources' if len(names) == len(sources) else ' + '.join(names)

for names in [('pictdes·WavLM', 'pataka·WavLM'), ('pictdes·WavLM', 'pictdes·ECAPA'), tuple(sources)]:
    X = concat(*names)
    run_cv(f'early: {label(names)}', fit_lr(X))
    print(f'{"":45s} {X.shape[1]} dimensions, {len(X)} participants')

## Dimension imbalance
After standardisation every dimension has unit variance, so a block with 768 dimensions has four times the "weight" of a block with 192 dimensions.
One remedy: reduce **each block separately** to the same number of dimensions (here PCA to 32), fitted on the training folds only.
`ColumnTransformer` applies a separate pipeline to every block, and because it is part of the pipeline, the PCA is refit in every fold.

In [ ]:
def blockwise_pca(names, n_components=32):
    def model():
        blocks = [(n, make_pipeline(StandardScaler(), PCA(n_components, random_state=0)),
                   [c for c in X.columns if c.startswith(f'{n}:')]) for n in names]
        return make_pipeline(ColumnTransformer(blocks), lr())
    X = concat(*names)
    return fit_lr(X, model)

for names in [('pictdes·WavLM', 'pictdes·ECAPA'), tuple(sources)]:
    run_cv(f'early, PCA per block: {label(names)}', blockwise_pca(names))

**Questions**
1. Does early fusion beat the best single source? If not, why not, given the cross tables from Part 4?
2. How many training participants per fold do we have, and how many dimensions after concatenating all four sources? What does that mean for overfitting?
3. Why must the PCA be fit inside the cross-validation, and what would go wrong if we ran it on all participants beforehand?
4. Early fusion uses fewer participants than the single-task models. When would that be a serious problem in practice?

# Part 6: Late Fusion (15 min)
Late fusion trains one classifier per source and averages their **probabilities**.
Our `combine` function averages over the sources that are **available** for a participant, so a missing task is no problem.

In [ ]:
def combine(probas, weights=None):
    # weighted average of probability tables; a participant missing in a source uses only the other sources
    weights = np.ones(len(probas)) if weights is None else np.asarray(weights, dtype=float)
    idx = probas[0].index
    for p in probas[1:]:
        idx = idx.union(p.index)
    P = np.stack([p.reindex(idx).to_numpy() for p in probas])         # (sources, participants, classes)
    available = ~np.isnan(P[..., :1])
    w = weights[:, None, None] * available
    w = np.where(w.sum(axis=0) > 0, w, available)    # only source available has weight 0: use it anyway
    return pd.DataFrame(np.nansum(P * w, axis=0) / w.sum(axis=0), index=idx, columns=CLASSES)


def late(names, weights=None):
    def fit_predict(train_ids, test_ids):
        return combine([fit_lr(sources[n])(train_ids, test_ids) for n in names], weights)
    return fit_predict

for names in [('pictdes·WavLM', 'pataka·WavLM'), ('pictdes·WavLM', 'pictdes·ECAPA'), tuple(sources)]:
    run_cv(f'late (mean): {label(names)}', late(names))

## Choosing the weights
Plain averaging gives every source the same weight.
It is tempting to try several weights and take the best one.
The plot below does exactly that with the out-of-fold probabilities from Part 4, i.e. it chooses the weight **on the test folds**.

In [ ]:
A, B = 'pictdes·WavLM', 'pataka·WavLM'
grid = np.linspace(0, 1, 21)
sweep = [uar(EVAL_IDS, combine([oof[A], oof[B]], [w, 1 - w])) for w in grid]

plt.figure(figsize=(7, 4))
plt.plot(grid, sweep, marker='o')
plt.axhline(1 / 3, color='gray', ls=':', label='chance')
plt.xlabel(f'weight of {A}  (weight of {B} = 1 - w)')
plt.ylabel('UAR (participants with both tasks)')
plt.title('Weight chosen on the test folds: optimistic!')
plt.legend()
plt.show()
print(f'best weight {grid[np.argmax(sweep)]:.2f}: UAR {max(sweep):.3f}')

The honest way: treat the weight as a hyperparameter and choose it **inside every training fold**.
We split the training participants once more (inner 5-fold CV), compute inner out-of-fold probabilities for both sources, choose the weight with the best inner UAR, retrain both models on the whole training fold and apply the weight to the test fold.

In [ ]:
INNER = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def inner_oof(name, train_ids):
    # out-of-fold probabilities within the training participants (one row per participant, so plain stratified k-fold is participant-safe)
    X = sources[name]
    ids = X.index.intersection(train_ids)
    P = cross_val_predict(lr(), X.loc[ids], y[ids], cv=INNER, method='predict_proba')
    return pd.DataFrame(P, index=ids, columns=CLASSES)


chosen_weights = []

def late_tuned(a, b):
    def fit_predict(train_ids, test_ids):
        pa, pb = inner_oof(a, train_ids), inner_oof(b, train_ids)
        both = pa.index.intersection(pb.index)
        w = grid[np.argmax([uar(both, combine([pa, pb], [w, 1 - w])) for w in grid])]
        chosen_weights.append(w)
        return combine([fit_lr(sources[a])(train_ids, test_ids), fit_lr(sources[b])(train_ids, test_ids)], [w, 1 - w])
    return fit_predict

run_cv(f'late (tuned weight): {A} + {B}', late_tuned(A, B))
print(f'weight of {A} chosen per fold:', [round(float(w), 2) for w in chosen_weights])

## A missing task is no problem
So far we evaluated on participants with both tasks.
Late fusion can also predict participants for whom only one task exists.

In [ ]:
for name in [f'early: {A} + {B}', f'late (mean): {A} + {B}']:
    p = oof[name]
    print(f'{name:45s} predicts {len(p):3d} participants, UAR on all of them = {uar(p.index, p):.3f}')
print(f'{"participants with at least one task":45s} {n_any:3d}')

**Questions**
1. Compare late with early fusion for the same sources. Which one works better here, and why could that be the case with so few participants?
2. Why is the maximum of the weight plot an optimistic estimate? How large is the gap to the honestly tuned weight?
3. Do the chosen weights agree across the folds? What does it tell you if they jump around?
4. What happens to the plain average if one source is much weaker than the other? Try the average of the best and the worst source from Part 4.
5. Why do we average probabilities and not predicted labels? What would majority voting with two sources do?

# Part 7: Stacking (10 min)
Stacking replaces the hand-chosen weights by a small **meta-classifier** that learns how to combine the probabilities of the base models.
The meta-classifier gets the probabilities of all sources (here 4 × 3 = 12 numbers per participant) and can learn, e.g., "trust `pataka` more when it is confident about AD".

The crucial detail: the meta-classifier must be trained on **out-of-fold** probabilities of the training participants (the inner CV from Part 6).
For comparison we also build a **leaky** version that trains the meta-classifier on the probabilities of base models that have seen the same participants.

In [ ]:
def meta():
    return LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced')


def stack_features(probas, ids):
    return np.hstack([p.loc[ids].to_numpy() for p in probas])


def stacking(names, leaky=False):
    def fit_predict(train_ids, test_ids):
        if leaky:     # WRONG: base models predict their own training participants
            train_probas = [fit_lr(sources[n])(train_ids, train_ids) for n in names]
        else:         # correct: inner out-of-fold probabilities
            train_probas = [inner_oof(n, train_ids) for n in names]
        test_probas = [fit_lr(sources[n])(train_ids, test_ids) for n in names]
        tr = train_probas[0].index
        te = test_probas[0].index
        for p in train_probas[1:]:
            tr = tr.intersection(p.index)
        for p in test_probas[1:]:
            te = te.intersection(p.index)
        clf = meta().fit(stack_features(train_probas, tr), y[tr])
        return pd.DataFrame(clf.predict_proba(stack_features(test_probas, te)), index=te, columns=CLASSES)
    return fit_predict

for names in [(A, B), tuple(sources)]:
    run_cv(f'stacking: {label(names)}', stacking(names))
run_cv('stacking LEAKY: all sources', stacking(tuple(sources), leaky=True));

**Questions**
1. How confident are the base models on participants they were trained on, compared with participants they have not seen? What does the leaky meta-classifier therefore learn, and why does that not transfer to the test fold?
2. Compare the correct and the leaky stacking result. Leakage can make the result look better *or* worse. Why is it a methodological error in both cases?
3. Stacking has more parameters than a weighted average. When is it worth it, and when should you prefer the simple average?
4. Stacking needs all sources for training the meta-classifier in our implementation. How could you handle a missing source?

# Part 8: Intermediate / Hybrid Fusion (10 min)
Instead of combining decisions, we can combine **learned representations**.
The small network below projects every source to 32 dimensions, computes a **gate** (a softmax weight per source and participant) from all projections, and classifies the gated sum.
It is trained end to end in every fold.
(A real hybrid model would also fine-tune the encoders, which is far too expensive here.)

Because the gate is computed per participant, we can look at how much the network relies on each source.

In [ ]:
class GatedFusion(nn.Module):
    def __init__(self, dims, hidden=32, n_classes=3):
        super().__init__()
        self.proj = nn.ModuleList([nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(0.3)) for d in dims])
        self.gate = nn.Linear(hidden * len(dims), len(dims))
        self.head = nn.Linear(hidden, n_classes)

    def forward(self, xs):
        h = torch.stack([p(x) for p, x in zip(self.proj, xs)], dim=1)       # (batch, sources, hidden)
        g = torch.softmax(self.gate(h.flatten(1)), dim=1)                    # (batch, sources)
        return self.head((g[..., None] * h).sum(dim=1)), g


gates = {}

def hybrid(names, seed=0, epochs=200):
    def fit_predict(train_ids, test_ids):
        X = concat(*names)
        tr, te = X.index.intersection(train_ids), X.index.intersection(test_ids)
        blocks = [[c for c in X.columns if c.startswith(f'{n}:')] for n in names]
        scalers = [StandardScaler().fit(X.loc[tr, b]) for b in blocks]          # training statistics only
        to_tensors = lambda ids: [torch.tensor(s.transform(X.loc[ids, b]), dtype=torch.float32) for s, b in zip(scalers, blocks)]
        torch.manual_seed(seed)
        net = GatedFusion([len(b) for b in blocks])
        opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
        counts = np.bincount(y[tr], minlength=3)
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(len(tr) / (3 * counts), dtype=torch.float32))
        x_tr, y_tr = to_tensors(tr), torch.tensor(y[tr].to_numpy())
        net.train()
        for _ in range(epochs):                         # full-batch training, no early stopping (fixed in advance)
            opt.zero_grad()
            loss_fn(net(x_tr)[0], y_tr).backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            logits, g = net(to_tensors(te))
        gates.setdefault(seed, []).append(pd.DataFrame(g.numpy(), index=te, columns=names))
        return pd.DataFrame(torch.softmax(logits, 1).numpy(), index=te, columns=CLASSES)
    return fit_predict


hybrid_uars = []
for seed in range(3):
    run_cv(f'hybrid (gated), seed {seed}', hybrid(tuple(sources), seed=seed))
    hybrid_uars.append(results.pop(f'hybrid (gated), seed {seed}')['UAR'])
results['hybrid (gated): all sources, mean of 3 seeds'] = {'UAR': np.mean(hybrid_uars), 'std': np.std(hybrid_uars), 'n predicted': len(EVAL_IDS)}
print(f'mean over seeds: {np.mean(hybrid_uars):.3f} ± {np.std(hybrid_uars):.3f} (std over seeds)')

pd.concat(gates[0]).mean().rename('mean gate weight (seed 0)')

**Questions**
1. How does the hybrid model compare with late fusion and stacking? How much do the seeds differ?
2. Which source gets the largest mean gate weight? Does it agree with the single-source ranking from Part 4?
3. The network has far more parameters than the logistic regressions. Which design decisions above try to limit overfitting, and what would you need to tune them honestly?
4. Hybrid fusion can model *interactions* between sources. Give an example of an interaction between `pictdes` and `pataka` that could be diagnostically relevant.

# Part 9: Summary and Discussion (10 min)

In [ ]:
table = pd.DataFrame(results).T.sort_values('UAR', ascending=False)
display(table.style.format({'UAR': '{:.3f}', 'std': '{:.3f}', 'n predicted': '{:.0f}'}))

fig, ax = plt.subplots(figsize=(8, 0.35 * len(table) + 1))
colors = ['tab:gray' if n in sources else 'tab:red' if 'LEAKY' in n else 'tab:blue' for n in table.index]
ax.barh(table.index, table['UAR'], xerr=table['std'], color=colors)
ax.axvline(1 / 3, color='k', ls=':', label='chance')
ax.axvline(max(results[n]['UAR'] for n in sources), color='tab:gray', ls='--', label='best single source')
ax.invert_yaxis()
ax.set_xlabel('UAR (mean ± std over 5 participant folds, participants with both tasks)')
ax.legend(loc='lower right')
plt.show()

In [ ]:
best_single = max(sources, key=lambda n: results[n]['UAR'])
best_fusion = max((n for n in oof if n not in sources and 'LEAKY' not in n and 'seed' not in n), key=lambda n: results[n]['UAR'])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name in zip(axes, [best_single, best_fusion]):
    pred = oof[name].loc[EVAL_IDS].to_numpy().argmax(1)
    ConfusionMatrixDisplay(confusion_matrix(y[EVAL_IDS], pred, normalize='true'), display_labels=CLASSES).plot(ax=ax, colorbar=False, values_format='.2f')
    ax.set_title(name, fontsize=9)
plt.tight_layout()
plt.show()

**Questions**
1. Which fusion strategy works best here, and does it clearly beat the best single source (compare the difference with the standard deviations)?
2. We picked the "best" systems by looking at the same cross-validation that we report. Why is this optimistic, and how would you report a final result in the PStA?
3. Look at the confusion matrices. Which class is hardest, and does fusion change *which* errors are made?
4. Rank the strategies by how much data they need and by how easy they are to explain to a clinician.

**Transfer to your PStA (audio + text)**
5. In the PStA you fuse audio with text (e.g. transcripts of `pictdes`). Which of today's observations do you expect to transfer, and what is different (dimensionality, missing transcripts, ASR errors)?
6. Which strategy would you choose as your first fusion model, and which one as the "advanced" model? Justify your choice with today's results.
7. Write down the evaluation protocol for your project in three sentences: folds, metric, how hyperparameters and fusion weights are chosen.